In [15]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [16]:

import sys

sys.path.append('../scripts')

In [17]:
import numpy as np
import scanpy as sc
import os
import matplotlib.pyplot as plt
import decoupler as dc
import scipy.sparse as sp
import pandas as pd

from cellina import make_neighbor_perturbation
from utils import set_seed
from train_loo import preprocess_crc, preprocess_merfish, _load_model, split_indices
from counterfactual_analysis import compute_lfc_metrics, compute_rmse, compute_edistance, mixing_index, _to_dense, _normalize_counts
from configs.adata_crc_config import ADATA_ARGS as ADATA_ARGS_CRC
from configs.adata_merfish_config import ADATA_ARGS as ADATA_ARGS_MERFISH

In [18]:
import cellina
cellina.__version__

'0.7.1'

In [19]:
set_seed(0)

In [20]:
DATASET_NAME = "merfish"  # or "merfish"
MODEL_ROOT = "/data/a330d/data/ood/trained"

In [21]:
CRC_PATHS = [
    #"/data2/a330d/datasets/crc/raw_zenodo/crc_210.h5ad",
    #"/data2/a330d/datasets/crc/raw_zenodo/crc_221.h5ad",
    #"/data2/a330d/datasets/crc/raw_zenodo/crc_231.h5ad",
    #"/data2/a330d/datasets/crc/raw_zenodo/crc_232.h5ad",
    "/data2/a330d/datasets/crc/raw_zenodo/crc_242.h5ad",
    "/data2/a330d/datasets/crc/raw_zenodo/crc_120.h5ad",
]

CRC_HOLDOUTS = [
    "Endothelial",
    "Epithelial",
    "Fibroblast",
    "Myeloid",
    "T_cell",
]

MERFISH_PATHS = [
    "/data/a330d/datasets/MERFISH_mouse_brain/C57BL6J-2.036.h5ad",    
    "/data/a330d/datasets/MERFISH_mouse_brain/C57BL6J-2.039.h5ad",
    "/data/a330d/datasets/MERFISH_mouse_brain/C57BL6J-2.041.h5ad",
]

MERFISH_HOLDOUTS = [
    'glutamatergic neuron',
    'oligodendrocyte',
    'astrocyte',
    'GABAergic neuron',
    'endothelial cell',
]

PATHS = CRC_PATHS if DATASET_NAME == "crc" else MERFISH_PATHS
HOLDOUT_CELLTYPES = CRC_HOLDOUTS if DATASET_NAME == "crc" else MERFISH_HOLDOUTS
DATA_ARGS = ADATA_ARGS_CRC if DATASET_NAME == "crc" else ADATA_ARGS_MERFISH

In [22]:
n_top_genes = DATA_ARGS.get('n_top_genes')
labels_key = DATA_ARGS.get('labels_key')
domains_key = DATA_ARGS.get('domains_key')
batch_key = DATA_ARGS.get('batch_key')
control_domain = DATA_ARGS.get('control_domains')[0]
holdout_domains = DATA_ARGS.get('holdout_domains')
n_neighbors = DATA_ARGS.get('n_neighbors')
batch_size = 2048
library_size = 'latent'
n_deg = 50
n_pert_genes = 200

In [23]:
# Create SLIDES which contain file names from PATHS - first split by "/" and take last part, then split by "." and take first part
SLIDES = [path.split("/")[-1].split(".h5ad")[0] for path in PATHS]

In [24]:
def get_perturbation_logfc(adata, control_domain, holdout_domain, labels_key, domains_key):
    # Cell-type-specific
    pdata_ct = dc.pp.pseudobulk(
        adata=adata, sample_col=domains_key, groups_col=labels_key, mode='sum', layer='counts'
    )
    sc.pp.normalize_total(pdata_ct, target_sum=1e4)
    sc.pp.log1p(pdata_ct)

    cell_types_with_both = [
        ct for ct in pdata_ct.obs[labels_key].unique()
        if ((pdata_ct.obs[domains_key] == control_domain) & (pdata_ct.obs[labels_key] == ct)).any()
        and ((pdata_ct.obs[domains_key] == holdout_domain) & (pdata_ct.obs[labels_key] == ct)).any()
    ]

    _ct_rows = []
    for _ct in cell_types_with_both:
        _crc_ct = pdata_ct[(pdata_ct.obs[domains_key] == holdout_domain) & (pdata_ct.obs[labels_key] == _ct)].X
        _ref_ct = pdata_ct[(pdata_ct.obs[domains_key] == control_domain) & (pdata_ct.obs[labels_key] == _ct)].X
        _crc_m  = np.asarray(_crc_ct.mean(axis=0)).flatten() if sp.issparse(_crc_ct) else _crc_ct.mean(axis=0).flatten()
        _ref_m  = np.asarray(_ref_ct.mean(axis=0)).flatten() if sp.issparse(_ref_ct) else _ref_ct.mean(axis=0).flatten()
        _ct_rows.append(pd.Series(_crc_m - _ref_m, index=pdata_ct.var_names, name=_ct))
    domain_logfc_df = pd.concat(_ct_rows, axis=1).T

    return domain_logfc_df

In [ ]:
results = []
for path, slide_id in zip(PATHS, SLIDES):
    adata = sc.read(path)
    
    if DATASET_NAME == 'crc':
        adata = preprocess_crc(adata, n_top_genes=n_top_genes, n_neighbors=n_neighbors, labels_key=labels_key, domains_key=domains_key)
    elif DATASET_NAME == 'merfish':
        adata = preprocess_merfish(adata, n_top_genes=n_top_genes, n_neighbors=n_neighbors, labels_key=labels_key, domains_key=domains_key)
    else:
        raise ValueError(f"Unknown dataset_name: {DATASET_NAME}. Supported: crc, merfish")
    

    for holdout_celltype in HOLDOUT_CELLTYPES:
        # 50 times * in print
        print(f"{'='*50} Slide: {slide_id}, Holdout Celltype: {holdout_celltype} {'='*50}")
        # create splits
        train_idx, val_idx, test_idx = split_indices(adata,
                                                    holdout_celltype,
                                                    labels_key=labels_key,
                                                    domains_key=domains_key,
                                                    holdout_domains=holdout_domains,
                                                    seed=0)

        splits = (train_idx, val_idx, test_idx)

        save_dir = os.path.join(MODEL_ROOT, slide_id, holdout_celltype, 'cellina')
        model = _load_model(save_dir,
                            model_class='cellina',
                            adata=adata,
                            splits=splits)
        is_control_region = adata.obs[domains_key]==(control_domain)
        is_holdout_ct = adata.obs[labels_key].astype(str) == holdout_celltype
        mask_control = is_control_region & is_holdout_ct
        idx_control = np.where(mask_control.values)[0]    

        for hd in holdout_domains:
                # ── Cell-type-specific perturbation ──────────────────────────────────────
                domain_logfc_df = get_perturbation_logfc(adata, control_domain, hd, labels_key, domains_key)
                logfc_series_dict = {}
                s = domain_logfc_df.loc[holdout_celltype]
                logfc_series_dict[holdout_celltype] = s[s.abs().nlargest(n_pert_genes).index]
                make_neighbor_perturbation(
                        adata, perturbations=logfc_series_dict, groupby=labels_key,
                        obsm_key_out='spatial_x_cf', base=np.e,
                )
                pert_expr = model.get_perturbed_expression(
                        adata=adata, indices=idx_control, spatial_obsm_key='spatial_x_cf',
                        batch_size=batch_size, library_size=library_size,
                        )
                
                is_holdout_region = adata.obs[domains_key].astype(str) == hd
                mask_target = is_holdout_region & is_holdout_ct
                idx_target = np.where(mask_target.values)[0]
                
                # Compute stats
                control = adata.layers['counts'][mask_control.values, :]
                target = adata.layers['counts'][mask_target.values, :]
                control, target = _to_dense(control), _to_dense(target)
                counterfactual = pert_expr

                pear, spear, prec, dir_match, deg = compute_lfc_metrics(control=control, target=target, counterfactual=counterfactual, n_deg=n_deg)
                rmse = compute_rmse(observed=target, predicted=counterfactual, deg=deg)
                edist_global = compute_edistance(adata, observed=target, predicted=counterfactual, deg=None, library_size=1e4)
                edist_local = compute_edistance(adata, observed=target, predicted=counterfactual, deg=None, library_size=1e4, local=True)
                edist_pca_log = compute_edistance(adata, observed=target, predicted=counterfactual, deg=None, library_size=1e4, local=True, use_pca=True)
                edist_pca = compute_edistance(adata, observed=target, predicted=counterfactual, deg=None, library_size=1e4, local=True, use_pca=True, log1p=False)
                mix_idx = mixing_index(observed=target, predicted=counterfactual, library_size=1e4)
                _, _, _, dir_match_k, _ = compute_lfc_metrics(control=control, target=target, counterfactual=counterfactual, n_deg=n_deg, direction_match_normalize="k")
                _, _, _, dir_match_gt, _ = compute_lfc_metrics(control=control, target=target, counterfactual=counterfactual, n_deg=n_deg, direction_match_normalize="gt_topk")

                results.append(
                        dict(
                        dataset_name=DATASET_NAME,
                        sid=slide_id,
                        control_domain=control_domain,
                        target_domain=hd,
                        n_deg=n_deg,
                        model_name="cellina-pert",
                        holdout_celltype=holdout_celltype,
                        spearman=spear,
                        pearson=pear,
                        precision=prec,
                        direction_match=dir_match,
                        direction_match_k=dir_match_k,
                        direction_match_gt=dir_match_gt,
                        mixing_index=mix_idx,
                        edistance_global=edist_global,
                        edistance_local=edist_local,
                        edistance_pca_log=edist_pca_log,
                        edistance_pca=edist_pca,
                        rmse=np.log10(rmse),
                        top_n_perturb=n_pert_genes,
                        )
                )

/data/a330d/projects/cellina-reproducibility/notebooks/../scripts/train_loo.py:175: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[labels_key] = adata.obs[labels_key].astype("category")
Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.


================================================== Slide: C57BL6J-2.036, Holdout Celltype: glutamatergic neuron ==================================================
INFO     File /data/a330d/data/ood/trained/C57BL6J-2.036/glutamatergic neuron/cellina/model.pt already downloaded  
INFO     cellina: The Cellina model has been initialized with adversarial domain forgetting                        
cellina loaded model from /data/a330d/data/ood/trained/C57BL6J-2.036/glutamatergic neuron/cellina


/data/a330d/miniforge3/envs/cellina-graph/lib/python3.11/site-packages/scvi/data/fields/_dataframe_field.py:227: UserWarning: Category 17 in adata.obs['_scvi_labels'] has fewer than 3 cells. Models may not train properly.
  new_mapping = _make_column_categorical(
/data/a330d/miniforge3/envs/cellina-graph/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)
/data/a330d/miniforge3/envs/cellina-graph/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)
Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.


================================================== Slide: C57BL6J-2.036, Holdout Celltype: oligodendrocyte ==================================================
INFO     File /data/a330d/data/ood/trained/C57BL6J-2.036/oligodendrocyte/cellina/model.pt already downloaded       
INFO     cellina: The Cellina model has been initialized with adversarial domain forgetting                        
cellina loaded model from /data/a330d/data/ood/trained/C57BL6J-2.036/oligodendrocyte/cellina


/data/a330d/miniforge3/envs/cellina-graph/lib/python3.11/site-packages/scvi/data/fields/_dataframe_field.py:227: UserWarning: Category 17 in adata.obs['_scvi_labels'] has fewer than 3 cells. Models may not train properly.
  new_mapping = _make_column_categorical(
/data/a330d/miniforge3/envs/cellina-graph/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)
/data/a330d/miniforge3/envs/cellina-graph/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)
Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.


...

/data/a330d/miniforge3/envs/cellina-graph/lib/python3.11/site-packages/scvi/data/fields/_dataframe_field.py:227: UserWarning: Category 16 in adata.obs['_scvi_labels'] has fewer than 3 cells. Models may not train properly.
  new_mapping = _make_column_categorical(
/data/a330d/miniforge3/envs/cellina-graph/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)
/data/a330d/miniforge3/envs/cellina-graph/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)
Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.


================================================== Slide: C57BL6J-2.041, Holdout Celltype: GABAergic neuron ==================================================
INFO     File /data/a330d/data/ood/trained/C57BL6J-2.041/GABAergic neuron/cellina/model.pt already downloaded      
INFO     cellina: The Cellina model has been initialized with adversarial domain forgetting                        
cellina loaded model from /data/a330d/data/ood/trained/C57BL6J-2.041/GABAergic neuron/cellina


/data/a330d/miniforge3/envs/cellina-graph/lib/python3.11/site-packages/scvi/data/fields/_dataframe_field.py:227: UserWarning: Category 16 in adata.obs['_scvi_labels'] has fewer than 3 cells. Models may not train properly.
  new_mapping = _make_column_categorical(
/data/a330d/miniforge3/envs/cellina-graph/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)
/data/a330d/miniforge3/envs/cellina-graph/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)
Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.


================================================== Slide: C57BL6J-2.041, Holdout Celltype: endothelial cell ==================================================
INFO     File /data/a330d/data/ood/trained/C57BL6J-2.041/endothelial cell/cellina/model.pt already downloaded      
INFO     cellina: The Cellina model has been initialized with adversarial domain forgetting                        
cellina loaded model from /data/a330d/data/ood/trained/C57BL6J-2.041/endothelial cell/cellina


/data/a330d/miniforge3/envs/cellina-graph/lib/python3.11/site-packages/scvi/data/fields/_dataframe_field.py:227: UserWarning: Category 16 in adata.obs['_scvi_labels'] has fewer than 3 cells. Models may not train properly.
  new_mapping = _make_column_categorical(
/data/a330d/miniforge3/envs/cellina-graph/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)
/data/a330d/miniforge3/envs/cellina-graph/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)


In [26]:
results

[{'dataset_name': 'merfish',
  'sid': 'C57BL6J-2.036',
  'control_domain': 'Thalamus',
  'target_domain': 'Isocortex',
  'n_deg': 50,
  'model_name': 'cellina-pert',
  'holdout_celltype': 'glutamatergic neuron',
  'spearman': np.float64(0.8743817527010804),
  'pearson': np.float32(0.9489304),
  'precision': 0.42,
  'direction_match': np.float64(1.0),
  'direction_match_k': np.float64(0.42),
  'dir_match_gt': np.float64(1.0),
  'mixing_index': np.float64(0.4479925049590252),
  'edistance_global': np.float64(38.683950424194336),
  'edistance_local': np.float64(46.2284553527832),
  'edist_pca_log': np.float64(13.497933387756348),
  'edist_pca': np.float64(527.1666198730469),
  'rmse': np.float32(4.1942825),
  'top_n_perturb': 200},
 {'dataset_name': 'merfish',
  'sid': 'C57BL6J-2.036',
  'control_domain': 'Thalamus',
  'target_domain': 'Fiber_tracts',
  'n_deg': 50,
  'model_name': 'cellina-pert',
  'holdout_celltype': 'glutamatergic neuron',
  'spearman': np.float64(0.825594237695078),
 

In [27]:
results_csv_name = f'../results/loo_cellina_{DATASET_NAME}_DEG_{n_deg}'
results_csv_path = results_csv_name + '_pert.csv'
df_results = pd.DataFrame(results)
if os.path.exists(results_csv_path):
    df_results.to_csv(f"{results_csv_path}", mode='a', header=False, index=False)
else:
    df_results.to_csv(f"{results_csv_path}", index=False)